# 27. Corrective RAG (CRAG)

Grade retrieval quality; refine good evidence, and fall back to web search when it is poor.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

## Why this matters

What if retrieval returns nothing useful? Basic RAG answers anyway. **Corrective RAG (CRAG)** adds a *retrieval evaluator* that grades the evidence as correct / ambiguous / incorrect and acts: refine good evidence (strip noise), combine ambiguous evidence with web search, or discard bad evidence and search the web instead.

## How it works

1. Retrieve top-k from the local index
2. Grade each chunk: correct / ambiguous / incorrect
3. **Correct** -> keep, refine to key sentences
4. **Ambiguous** -> keep + add web search results
5. **Incorrect** -> ignore local results, use web search only
6. Generate from the corrected context

```text
 retrieve --> grade --+-- correct   --> refine ------------------\
                      +-- ambiguous --> refine + web search ------+--> generate
                      +-- incorrect --> web search only ---------/
```

## Prerequisites

Needs a running Ollama server (`ollama serve`) with `llama3.2` pulled, plus the shared helpers in `../common.py`.

**Extra requirements for this notebook:** `pip install ddgs`

## Setup

Load the handbook, split it into ~500-character chunks (50 overlap) and index them. Every notebook starts from this same baseline so results are comparable.

In [ ]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

In [ ]:
def grade(query, chunk, similarity):
    """Retrieval evaluator: the LLM's verdict combined with the vector similarity.
    LLM yes + decent similarity -> correct; one signal only -> ambiguous; neither -> incorrect."""
    llm_ok = llm_yes(f"Question: {query}\n\nPassage: {chunk}\n\nDoes the passage contain information that helps answer the question?")
    sim_ok = similarity >= 0.30
    return "correct" if llm_ok and sim_ok else "ambiguous" if llm_ok or sim_ok else "incorrect"

def refine(query, chunk):
    """Knowledge refinement: keep only the useful sentences."""
    return llm(f"Copy only the sentences relevant to the question, verbatim.\n\nQuestion: {query}\n\nPassage: {chunk}")

def web_search(query, n=3):
    try:
        try:
            from ddgs import DDGS
        except ImportError:
            from duckduckgo_search import DDGS
        return [r["body"] for r in DDGS().text(query, max_results=n)]
    except Exception as e:
        print("web search unavailable:", e)
        return []

In [ ]:
def crag(query):
    hits = store.search(query, 4)
    grades = [grade(query, h["text"], h["score"]) for h in hits]
    good = [refine(query, h["text"]) for h, g in zip(hits, grades) if g == "correct"]
    if good:
        action, contexts = "use retrieved", good
    elif "ambiguous" in grades:
        rewritten = llm(f"Rewrite as a short web search query: {query}")
        action, contexts = "retrieved + web", [h["text"] for h, g in zip(hits, grades) if g == "ambiguous"] + web_search(rewritten)
    else:
        action, contexts = "web search only", web_search(llm(f"Rewrite as a short web search query: {query}"))
    return action, grades, (answer(query, contexts) if contexts else "No evidence found.")

for q in ["What is HNSW?", "Who is the current prime minister of Japan?"]:
    action, grades, response = crag(q)
    print(f"Q: {q}\n  grades={grades}\n  action={action}\n  A: {response[:300]}\n")

## Grade distribution on in-domain vs out-of-domain questions
In-domain questions should be mostly `correct`; out-of-domain mostly `incorrect`.

In [ ]:
import pandas as pd
probes = {"in-domain": ["What is HNSW?", "What is mean reciprocal rank?", "What does a reranker do?"],
          "out-of-domain": ["Who is the current prime minister of Japan?", "What is the tallest mountain in the world?", "Who wrote Hamlet?"]}
rows = []
for kind, qs_ in probes.items():
    for q in qs_:
        g = [grade(q, h["text"], h["score"]) for h in store.search(q, 4)]
        rows.append({"type": kind, "question": q, "grades": g})
pd.DataFrame(rows)

## Offline behaviour
If the web is unreachable the fallback returns nothing; CRAG should then say it cannot answer rather than guess.

In [ ]:
print("web results available:", len(web_search("Python programming language")) > 0)

## Compare against plain RAG

In [ ]:
compare_pipelines({"plain RAG": lambda q: rag(q, store, 3)[0], "CRAG": lambda q: crag(q)[2]}, n=4)

## Trade-offs

| Strengths | Weaknesses |
|---|---|
| Handles retrieval failure explicitly | Multiple LLM calls per query |
| Web fallback extends knowledge beyond the index | Web results are untrusted input (prompt-injection risk) |
| Refinement removes noise | Needs network access |

## When to use it

Open-domain assistants where the local index is incomplete or stale.

## Experiments to try

- Use your own search API instead of DuckDuckGo.
- Add confidence thresholds with numeric scores.
- Sanitise web snippets before prompting.

## Key takeaways

- Evaluate the retrieval, not just the answer.
- Three outcomes, three actions.
- Treat web content as untrusted.

---
**Next:** [28. Self-RAG](../28_self_rag/self_rag.ipynb)